In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zlatan599/garbage-dataset-classification")

print("Path to dataset files:", path)

In [ ]:
!ls {path}

In [ ]:
dataset_path = path + "/Garbage_Dataset_Classification/images"
!ls {dataset_path}

## 1. โหลด EfficientNet-B2 พร้อม pretrained weights และเปลี่ยน classifier
### pretrained-model-builder 

In [ ]:
from typing import Any
import torch
from torchvision.models import get_model, get_model_weights

import torch.nn as nn
from typing import Optional, Tuple


def create_pretrained_model(
    model_name: str,
    num_classes: int,
    freeze_backbone: bool = True,
    dropout: float = 0.2,
    seed: Optional[int] = 42,
) -> Tuple[nn.Module, Any]:
    """Creates a pretrained model, freezes the backbone, replaces the classifier head,

    and returns (model, transforms).
    """
    model_name_clean = model_name.strip().lower()

    # 1. Load weights and transforms
    weights_enum = get_model_weights(model_name_clean).DEFAULT
    model = get_model(model_name_clean, weights=weights_enum)
    transforms = weights_enum.transforms()

    # 2. Freeze backbone parameters
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # 3. Deterministic head replacement
    if seed is not None:
        torch.manual_seed(seed)

    # 4. Adapt classifier head for EfficientNet
    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):
        last_idx = len(model.classifier) - 1
        for idx in reversed(range(len(model.classifier))):
            if isinstance(model.classifier[idx], nn.Linear):
                last_idx = idx
                break

        in_features = model.classifier[last_idx].in_features
        model.classifier[last_idx] = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features=in_features, out_features=num_classes),
        )
    elif hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features=in_features, out_features=num_classes)
    else:
        raise NotImplementedError(f"Classifier adaptation not mapped for '{model_name}'.")

    return model, transforms


# Create EfficientNet-B2 with 2 output classes
model, transform = create_pretrained_model(
    model_name="efficientnet_b2",
    num_classes=6,
    freeze_backbone=True,
    dropout=0.2,
)

print(f"Model architecture: efficientnet_b2")
print(f"Base transforms:\n{transform}")

In [ ]:
model

In [ ]:
import subprocess
import sys
from typing import Sequence


try:
    from torchinfo import summary
    print("✅ Successfully imported summary from torchinfo.")
except ImportError:
    print("📦 'torchinfo' is not installed. Installing now...")
    try:
        get_ipython().system("pip install torchinfo")
        from torchinfo import summary
    except NameError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "torchinfo"])
    print("✅ Successfully imported summary from torchinfo.")


def inspect_model(
    model: nn.Module,
    input_size: Tuple[int, ...] = (1, 3, 288, 288),
    col_names: Sequence[str] = ("input_size", "output_size", "num_params", "trainable"),
    depth: int = 3,
    verbose: int = 1,
):
    """Summarizes a PyTorch model ensuring correct device allocation and input shape."""
    device = next(model.parameters()).device if list(model.parameters()) else "cpu"
    model.eval()

    return summary(
        model=model,
        input_size=input_size,
        device=device,
        col_names=list(col_names),
        depth=depth,
    )


inspect_model(model)

## 2. สร้าง Dataset, DataLoader และ Image Transform
### dataset-dataloader-builder

In [ ]:
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import datasets
from pathlib import Path
from typing import List, Tuple, Union

class DatasetFromSubset(Dataset):
    """Wrapper to apply transforms on subsets derived from random_split."""

    def __init__(self, subset: Subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx: int):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self) -> int:
        return len(self.subset)


def create_dataloaders(
    dataset_path: Union[str, Path],
    transform=transform,
    batch_size: int = 32,
    train_split: float = 0.8,
    val_split: float = 0.1,
    test_split: float = 0.1,
    num_workers: int = 2,
    seed: int = 42,
) -> Tuple[DataLoader, DataLoader, DataLoader, List[str]]:
    """Builds train, val, and test DataLoaders (80/10/10 split) using the specified transform."""
    data_path = Path(dataset_path)
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset path not found: {data_path}")

    raw_dataset = datasets.ImageFolder(data_path, transform=None)
    class_names = raw_dataset.classes

    total_size = len(raw_dataset)
    val_size = int(total_size * val_split)
    test_size = int(total_size * test_split)
    train_size = total_size - val_size - test_size

    generator = torch.Generator().manual_seed(seed)
    train_sub, val_sub, test_sub = random_split(
        raw_dataset, [train_size, val_size, test_size], generator=generator
    )

    train_data = DatasetFromSubset(train_sub, transform=transform)
    val_data = DatasetFromSubset(val_sub, transform=transform)
    test_data = DatasetFromSubset(test_sub, transform=transform)

    train_dataloader = DataLoader(
        train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
    )

    val_dataloader = DataLoader(
        val_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
    )

    test_dataloader = DataLoader(
        test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_dataloader, val_dataloader, test_dataloader, class_names, raw_dataset


train_loader, val_loader, test_loader, class_names, raw_dataset = create_dataloaders(
    dataset_path=dataset_path,
    transform=transform,
    train_split=0.8,
    val_split=0.1,
    test_split=0.1,
)

# กระดาษแข็ง แก้ว โลหะ พลาสติก กระดาษ ขยะ
print(f"Classes: {class_names}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

In [ ]:
print(f"test dataset {len(test_loader.dataset)} ")

### test-loader-augmented-visualizer

In [ ]:
import matplotlib.pyplot as plt


def visualize_train_loader_samples(
    train_loader: DataLoader,
    class_names: Optional[List[str]] = None,
    num_samples: int = 12,
    cols: int = 4,
    mean: Tuple[float, float, float] = (0.485, 0.456, 0.406),
    std: Tuple[float, float, float] = (0.229, 0.224, 0.225),
    figsize: Optional[Tuple[int, int]] = None,
) -> None:
    """Visualizes transformed/augmented images directly from train_loader."""
    collected_images = []
    collected_labels = []

    # 1. Fetch images from the DataLoader iterator
    for images, labels in train_loader:
        collected_images.append(images)
        collected_labels.append(labels)
        if sum(x.size(0) for x in collected_images) >= num_samples:
            break

    if not collected_images:
        print("⚠️ Warning: No batches retrieved from train_loader.")
        return

    # Concatenate and slice to exact sample size
    batch_images = torch.cat(collected_images, dim=0)[:num_samples]
    batch_labels = torch.cat(collected_labels, dim=0)[:num_samples]

    rows = (num_samples + cols - 1) // cols
    if figsize is None:
        figsize = (cols * 3.5, rows * 3.5)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if num_samples > 1 else [axes]

    # Pre-calculate un-normalization vectors
    mean_tensor = torch.tensor(mean).view(-1, 1, 1)
    std_tensor = torch.tensor(std).view(-1, 1, 1)

    for idx in range(num_samples):
        img = batch_images[idx].detach().cpu()
        label_idx = batch_labels[idx].item()

        # 2. Revert normalization (if 3-channel RGB image)
        if img.ndim == 3 and img.shape[0] == 3:
            img = img * std_tensor + mean_tensor
            img = torch.clamp(img, 0.0, 1.0)
            img_np = img.permute(1, 2, 0).numpy()
            cmap = None
        elif img.ndim == 3 and img.shape[0] == 1:
            img_np = img.squeeze(0).numpy()
            cmap = "gray"
        else:
            img_np = img.numpy()
            cmap = None

        # 3. Resolve class title
        if class_names and 0 <= label_idx < len(class_names):
            title = f"{class_names[label_idx]} (idx: {label_idx})"
        else:
            title = f"Class: {label_idx}"

        axes[idx].imshow(img_np, cmap=cmap)
        axes[idx].set_title(title, fontsize=10, fontweight="bold")
        axes[idx].axis("off")

    # Turn off leftover unused axes
    for j in range(num_samples, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Transformed & Augmented Samples from train_loader", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()


visualize_train_loader_samples(test_loader, class_names=class_names)

## 3. ประเมินโมเดลกับชุดทดสอบ
### model-test-evaluator

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import List, Optional, Tuple, Union
import random
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def evaluate_test_model(
    model: nn.Module,
    test_dataloader: DataLoader,
    class_names: List[str],
    loss_fn: Optional[nn.Module] = None,
    device: Optional[Union[str, torch.device]] = None,
) -> Tuple[float, int, int, np.ndarray, np.ndarray]:
    """Evaluates model performance over the test DataLoader."""
    if device is None:
        device = next(model.parameters()).device
    else:
        model = model.to(device)

    model.eval()
    total_correct = 0
    total_samples = 0
    running_loss = 0.0

    all_preds_list = []
    all_targets_list = []

    with torch.inference_mode():
        for X, y in test_dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)

            if loss_fn is not None:
                loss = loss_fn(outputs, y)
                running_loss += loss.item() * X.size(0)

            if outputs.ndim > 1 and outputs.shape[-1] > 1:
                preds = torch.argmax(outputs, dim=1)
            else:
                preds = (torch.sigmoid(outputs.squeeze(-1)) >= 0.5).long()

            total_correct += (preds == y).sum().item()
            total_samples += X.size(0)

            all_preds_list.extend(preds.cpu().numpy())
            all_targets_list.extend(y.cpu().numpy())

    all_preds = np.array(all_preds_list)
    all_targets = np.array(all_targets_list)
    test_acc = (total_correct / total_samples) * 100.0

    print("=" * 60)
    print("📊 TEST EVALUATION SUMMARY")
    print("=" * 60)
    if loss_fn is not None:
        print(f"Test Loss        : {running_loss / total_samples:.4f}")
    print(f"Total Samples    : {total_samples:,}")
    print(f"Correct Count    : {total_correct:,} / {total_samples:,}")
    print(f"Test Accuracy    : {test_acc:.2f}%")
    print("=" * 60)

    print("\nClassification Report:\n")
    print(classification_report(all_targets, all_preds, target_names=class_names))

    return test_acc, total_correct, total_samples, all_preds, all_targets


def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str],
    normalize: Optional[str] = None,
    figsize: Tuple[int, int] = (6, 5),
    cmap: str = "Blues",
) -> None:
    """Plots a labeled confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred, normalize=normalize)
    fmt = ".2f" if normalize else "d"

    plt.figure(figsize=figsize)
    sns.heatmap(
        cm,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=True,
        square=True,
    )
    plt.title(f"Confusion Matrix {'(Normalized)' if normalize else ''}", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Predicted Label", fontsize=11)
    plt.ylabel("True Label", fontsize=11)
    plt.tight_layout()
    plt.show()


def plot_random_predictions(
    model: nn.Module,
    test_dataloader: DataLoader,
    class_names: List[str],
    num_images: int = 12,
    cols: int = 4,
    mean: Tuple[float, float, float] = (0.485, 0.456, 0.406),
    std: Tuple[float, float, float] = (0.229, 0.224, 0.225),
    seed: Optional[int] = 42,
) -> None:
    """Samples and visualizes random test predictions with un-normalized images."""
    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)

    device = next(model.parameters()).device
    model.eval()

    collected_images = []
    collected_targets = []

    for X, y in test_dataloader:
        collected_images.append(X)
        collected_targets.append(y)
        if sum(len(x) for x in collected_images) >= num_images * 3:
            break

    all_X = torch.cat(collected_images, dim=0)
    all_y = torch.cat(collected_targets, dim=0)

    indices = random.sample(range(len(all_X)), min(num_images, len(all_X)))
    sample_images = all_X[indices]
    sample_targets = all_y[indices]

    with torch.inference_mode():
        logits = model(sample_images.to(device))
        preds = torch.argmax(logits, dim=1).cpu()
        probs = torch.softmax(logits, dim=1).cpu()

    rows = (len(indices) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = axes.flatten() if len(indices) > 1 else [axes]

    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)

    for i, idx in enumerate(range(len(indices))):
        img = sample_images[idx].cpu()
        img = img * std_tensor + mean_tensor
        img = torch.clamp(img, 0.0, 1.0)
        img = img.permute(1, 2, 0).numpy()

        true_label = class_names[sample_targets[idx].item()]
        pred_label = class_names[preds[idx].item()]
        conf = probs[idx][preds[idx]].item() * 100.0

        is_correct = preds[idx].item() == sample_targets[idx].item()
        color = "green" if is_correct else "red"

        axes[i].imshow(img)
        axes[i].set_title(
            f"True: {true_label}\nPred: {pred_label} ({conf:.1f}%)",
            color=color,
            fontsize=10,
            fontweight="bold",
        )
        axes[i].axis("off")

    for j in range(len(indices), len(axes)):
        axes[j].axis("off")

    plt.suptitle("Random Test Predictions (Green: Correct, Red: Incorrect)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Run evaluation on test loader
test_acc, total_correct, total_samples, all_preds, all_targets = evaluate_test_model(
    model=model,
    test_dataloader=test_loader,
    class_names=class_names,
    loss_fn=loss_fn,
)

# 2. Confusion matrix
plot_confusion_matrix(all_targets, all_preds, class_names=class_names)

# 3. Random prediction preview
plot_random_predictions(
    model=model,
    test_dataloader=test_loader,
    class_names=class_names,
    num_images=24,
    cols=4,
)

# กระดาษแข็ง แก้ว โลหะ พลาสติก กระดาษ ขยะ

## 4. โหลด model 
### model-artifact-saver

In [ ]:
!ls 

In [ ]:
def load_saved_artifacts(
    model: nn.Module,
    model_filename: str = "efficientnet_b2_garbage.pth",
    classes_filename: str = "class_names.txt",
    transform_filename: Optional[str] = "transform.pth",
    load_dir: Optional[Union[str, Path]] = None,
    device: Optional[Union[str, torch.device]] = None,
):
    """Loads saved model state_dict, class names, and transform from disk."""
    source_dir = Path(load_dir) if load_dir is not None else Path.cwd()
    model_path = source_dir / model_filename
    classes_path = source_dir / classes_filename

    if not model_path.exists():
        raise FileNotFoundError(f"Model weight file not found at: {model_path}")
    if not classes_path.exists():
        raise FileNotFoundError(f"Class names file not found at: {classes_path}")

    # 1. Load class labels
    with open(classes_path, mode="r", encoding="utf-8") as f:
        loaded_classes = [line.strip() for line in f if line.strip()]

    # 2. Determine target device
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 3. Restore state_dict into model
    model.load_state_dict(torch.load(f=str(model_path), map_location=device, weights_only=True))
    model.to(device)
    model.eval()

    # 4. Load transform if exists
    loaded_transform = None
    if transform_filename:
        transform_path = source_dir / transform_filename
        if transform_path.exists():
            loaded_transform = torch.load(f=str(transform_path), map_location="cpu", weights_only=False)
            print(f"✅ Loaded transform from: {transform_path}")

    print(f"✅ Loaded model weights from: {model_path}")
    print(f"✅ Loaded {len(loaded_classes)} classes from: {classes_path}")

    return model, loaded_classes, loaded_transform


# Load saved artifacts
model, class_names, loaded_transform = load_saved_artifacts(
    model=model,
    model_filename="efficientnet_b2_garbage.pth",
    classes_filename="class_names.txt",
    transform_filename="transform.pth",
)

## 5. ประเมินโมเดลที่โหลดมากับชุดทดสอบ
### model-test-evaluator

In [ ]:
# 1. Run evaluation on test loader
test_acc, total_correct, total_samples, all_preds, all_targets = evaluate_test_model(
    model=model,
    test_dataloader=test_loader,
    class_names=class_names,
    loss_fn=loss_fn,
)

# 2. Confusion matrix
plot_confusion_matrix(all_targets, all_preds, class_names=class_names)

# 3. Random prediction preview
plot_random_predictions(
    model=model,
    test_dataloader=test_loader,
    class_names=class_names,
    num_images=24,
    cols=4,
)

# กระดาษแข็ง แก้ว โลหะ พลาสติก กระดาษ ขยะ

## 6. สร้าง Gradio อินเทอร์เฟซ
### gradio-interface-builder

In [ ]:
import gradio as gr
from PIL import Image
from typing import Dict, Optional

device = next(model.parameters()).device if list(model.parameters()) else torch.device("cpu")
model.to(device)
model.eval()

# Use loaded_transform if available, otherwise fallback to transform
inference_transform = loaded_transform if loaded_transform is not None else transform


def predict(image: Optional[Image.Image]) -> Dict[str, float]:
    """Runs inference on a PIL image and returns class probabilities."""
    if image is None:
        return {}

    # 1. Preprocess image
    img_rgb = image.convert("RGB")
    tensor = inference_transform(img_rgb).unsqueeze(dim=0).to(device)

    # 2. Forward pass
    with torch.inference_mode():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze(0)

    # 3. Format predictions dictionary
    return {
        class_names[i]: float(probs[i].item())
        for i in range(len(class_names))
    }


def create_gradio_app(
    title: str = "🗑️ Garbage Dataset Classification",
    description: str = "อัปโหลดภาพขยะเพื่อจำแนกประเภท (cardboard, glass, metal, paper, plastic, trash) ด้วย EfficientNet-B2",
) -> gr.Blocks:
    """Builds a complete Gradio Blocks application."""
    with gr.Blocks(theme=gr.themes.Soft(), title=title) as demo:
        gr.Markdown(f"# {title}")
        gr.Markdown(description)

        with gr.Row():
            with gr.Column(scale=1):
                input_image = gr.Image(type="pil", label="Input Image")
                with gr.Row():
                    btn_clear = gr.ClearButton(components=[input_image], value="Clear")
                    btn_submit = gr.Button("Predict", variant="primary")

            with gr.Column(scale=1):
                output_label = gr.Label(num_top_classes=len(class_names), label="Top Predictions")

        # Interactive & button submission events
        btn_submit.click(fn=predict, inputs=input_image, outputs=output_label)
        input_image.change(fn=predict, inputs=input_image, outputs=output_label)

    return demo


app = create_gradio_app()
app.launch(share=True)

## 7. เตรียมการสำหรับการ deploy ไปยัง hf-spaces
### hf-spaces-deployment-prep

In [ ]:
import json
import shutil
from pathlib import Path
from torchvision.models import efficientnet_b2
from pathlib import Path
from PIL import Image
import torch
import gradio as gr
from torchvision import transforms
#from model import create_model

# 1. Setup deployment directory structure
deploy_dir = Path("hf_space_deployment")
weights_dir = deploy_dir / "weights"
assets_dir = deploy_dir / "assets"

deploy_dir.mkdir(parents=True, exist_ok=True)
weights_dir.mkdir(parents=True, exist_ok=True)
assets_dir.mkdir(parents=True, exist_ok=True)

# 2. Export Model Weights & Metadata
weights_path = weights_dir / "model_weights.pt"
torch.save(model.state_dict(), weights_path)
print(f"✅ Saved weights to {weights_path}")

classes_path = deploy_dir / "class_names.json"
with open(classes_path, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)
print(f"✅ Saved classes to {classes_path}")

# 3. Export sample images for demo examples
sample_count = 0
for cls in class_names:
    cls_folder = Path(dataset_path) / cls
    img_files = list(cls_folder.glob("*.jpg")) + list(cls_folder.glob("*.png"))
    if img_files:
        shutil.copy(img_files[0], assets_dir / f"{cls}_example.jpg")
        sample_count += 1
print(f"✅ Saved {sample_count} sample images to {assets_dir}")

# 4. Write README.md with Hugging Face Space YAML frontmatter
readme_content = """---
title: Garbage Classification (EfficientNet-B2)
emoji: 🗑️
colorFrom: green
colorTo: emerald
sdk: gradio
sdk_version: 4.44.0
app_file: app.py
pinned: false
license: mit
---

# Garbage Classification Demo
Predicts trash categories (cardboard, glass, metal, paper, plastic, trash) using a fine-tuned EfficientNet-B2.
"""
(deploy_dir / "README.md").write_text(readme_content, encoding="utf-8")
print("✅ Saved README.md")

# 5. Write requirements.txt
requirements_content = """torch
torchvision
gradio>=4.44.0
pillow
numpy
"""
(deploy_dir / "requirements.txt").write_text(requirements_content, encoding="utf-8")
print("✅ Saved requirements.txt")

# 6. Write standalone model.py
model_py_content = """import torch
import torch.nn as nn


def create_model(num_classes: int = 6, dropout: float = 0.2) -> nn.Module:
    model = efficientnet_b2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Sequential(
            nn.Dropout(p=dropout, inplace=False),
            nn.Linear(in_features=in_features, out_features=num_classes),
        ),
    )
    return model
"""
(deploy_dir / "model.py").write_text(model_py_content, encoding="utf-8")
print("✅ Saved model.py")

# 7. Write production app.py
app_py_content = """import json

DEPLOY_DIR = Path(__file__).parent

# Load class names
with open(DEPLOY_DIR / "class_names.json", "r", encoding="utf-8") as f:
    class_names = json.load(f)

# Device & Model
DEVICE = torch.device("cpu")
model = create_model(num_classes=len(class_names))
weights_path = DEPLOY_DIR / "weights" / "model_weights.pt"

if weights_path.exists():
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE, weights_only=True))
model.to(DEVICE)
model.eval()

# Transforms matching EfficientNet-B2 default evaluation preset (size 288)
transform = transforms.Compose([
    transforms.Resize(288, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(288),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict(image: Image.Image):
    if image is None:
        return {}
    img_rgb = image.convert("RGB")
    tensor = transform(img_rgb).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze(0)

    return {class_names[i]: float(probs[i].item()) for i in range(len(class_names))}

# Sample examples
example_images = [[str(p)] for p in sorted((DEPLOY_DIR / "assets").glob("*.jpg"))]

demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload Trash Image"),
    outputs=gr.Label(num_top_classes=len(class_names), label="Predictions"),
    examples=example_images if example_images else None,
    title="🗑️ Garbage Dataset Classification",
    description="Classify waste into cardboard, glass, metal, paper, plastic, or trash using EfficientNet-B2.",
)

if __name__ == "__main__":
    demo.launch()
"""
(deploy_dir / "app.py").write_text(app_py_content, encoding="utf-8")
print("✅ Saved app.py")

print("\n🚀 HF Spaces deployment files generated successfully inside: ./hf_space_deployment")

### Zip Deployment Directory

In [ ]:
# Archive hf_space_deployment folder into hf_space_deployment.zip
zip_path = shutil.make_archive(
    base_name="hf_space_deployment",
    format="zip",
    root_dir="hf_space_deployment",
)

print(f"✅ Archive created successfully: {zip_path}")